In [1]:
# Model that tests last 2 H4 bar correlation with next day
# Answer is that, at least for the stuff tested that context matters and its not that simple for H4. 
# For Gold its a bit more of a factor
# Does this apply to Daily, Weekly?
#Same as above really. More for Gold for revert, NAS, Dow seem to be trend follows. Weak stat signficance so context!
import pandas as pd
filtered_df = pd.DataFrame({'Currency': ['USDJPY', 'EURUSD','US30.a','XAUUSD.a','NAS100.a']})

C:\Users\nitis\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
#Test if US H4 bars affect the first two the next day or the next whole day. Answer not really

def run_hypothesis_test(csv_path):
    """
    Bulletproof pipeline updated with automatic delimiter detection
    to prevent out-of-bounds IndexErrors on tab/semicolon separated files.
    """
    print(f"Loading data from: {csv_path}")
    
    # Use sep=None and engine='python' to let pandas auto-detect if it's tab, comma, or semicolon
    try:
        raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    except Exception as e:
        print(f"Failed to read file {csv_path}: {e}")
        return None
        
    if raw_df.shape[1] < 2:
        print(f"⚠️ Warning: Only detected {raw_df.shape[1]} column. Forcing tab-separation fallback...")
        raw_df = pd.read_csv(csv_path, sep='\t')
        
    df = pd.DataFrame()
    
    # Force rename columns using position to strip away any '<DATE>' or '<TIME>' issues
    df['Date_Raw'] = raw_df.iloc[:, 0].astype(str)
    df['Time_Raw'] = raw_df.iloc[:, 1].astype(str)
    
    # ... Rest of the script remains exactly the same ...
    
    # Dynamically find OHLC columns by searching case-insensitive headers
    col_mapping = {c.lower().strip().replace('<','').replace('>',''): i for i, c in enumerate(raw_df.columns)}
    
    try:
        df['Open'] = raw_df.iloc[:, col_mapping['open']].astype(float)
        df['High'] = raw_df.iloc[:, col_mapping['high']].astype(float)
        df['Low'] = raw_df.iloc[:, col_mapping['low']].astype(float)
        df['Close'] = raw_df.iloc[:, col_mapping['close']].astype(float)
    except KeyError as e:
        print(f"Error mapping OHLC headers automatically. Detected headers: {list(raw_df.columns)}")
        print("Defaulting to standard positional slots (2=Open, 3=High, 4=Low, 5=Close)...")
        df['Open'] = raw_df.iloc[:, 2].astype(float)
        df['High'] = raw_df.iloc[:, 3].astype(float)
        df['Low'] = raw_df.iloc[:, 4].astype(float)
        df['Close'] = raw_df.iloc[:, 5].astype(float)

    # Clean the '2022.01.28' dot notation format securely
    df['Clean_Date'] = df['Date_Raw'].str.replace('.', '-', regex=False)
    
    # Create the unified datetime timeline index
    df['Timestamp'] = pd.to_datetime(df['Clean_Date'] + ' ' + df['Time_Raw'])
    df = df.sort_values(by='Timestamp').reset_index(drop=True)
    
    # Extract operational day components
    df['Date_Group'] = df['Timestamp'].dt.date
    df['Day_of_Week'] = df['Timestamp'].dt.dayofweek
    
    grouped = df.groupby('Date_Group')
    unique_dates = list(grouped.groups.keys())
    
    day_pairs = []
    
    print("Processing day-to-day transitions...")
    for i in range(len(unique_dates) - 1):
        day_t = unique_dates[i]
        day_t_plus_1 = unique_dates[i+1]
        
        df_t = grouped.get_group(day_t)
        df_t1 = grouped.get_group(day_t_plus_1)
        
        # Ensure we have minimum data density for processing 
        if len(df_t) >= 2 and len(df_t1) >= 2:
            last_2_bars = df_t.tail(2)
            first_2_bars = df_t1.head(2)
            
            # Feature calculation: Return across last two bars of Day T
            h4_last_2_return = (last_2_bars['Close'].iloc[-1] - last_2_bars['Open'].iloc[0]) / last_2_bars['Open'].iloc[0]
            
            # Target 1: Next-day open session tracking
            tgt_early_return = (first_2_bars['Close'].iloc[-1] - first_2_bars['Open'].iloc[0]) / first_2_bars['Open'].iloc[0]
            
            # Target 2: Complete next-day aggregate return
            tgt_full_return = (df_t1['Close'].iloc[-1] - df_t1['Open'].iloc[0]) / df_t1['Open'].iloc[0]
            
            day_pairs.append({
                'Date_T': day_t,
                'Date_T1': day_t_plus_1,
                'Predictor_Return': h4_last_2_return,
                'Target_Early_Return': tgt_early_return,
                'Target_Full_Return': tgt_full_return,
                'Is_Weekend': 1 if df_t['Day_of_Week'].iloc[0] == 4 else 0
            })
            
    analysis_df = pd.DataFrame(day_pairs)
    
    if len(analysis_df) == 0:
        print("Execution halted: No matching chronological day sequences could be stitched together.")
        return None
        
    print("\n" + "="*50)
    print("        HISTORICAL HYPOTHESIS TEST RESULTS        ")
    print("="*50)
    print(f"Total trading day transitions analyzed: {len(analysis_df)}")
    
    # Test A: Linear Correlation Evaluation
    corr_early, p_corr_early = stats.pearsonr(analysis_df['Predictor_Return'], analysis_df['Target_Early_Return'])
    corr_full, p_corr_full = stats.pearsonr(analysis_df['Predictor_Return'], analysis_df['Target_Full_Return'])
    
    print("\n[1] LINEAR CORRELATION ANALYSIS (Pearson r)")
    print(f" -> Day T Last 2 Bars vs Next Day First 2 Bars: r = {corr_early:.4f} (p-value: {p_corr_early:.4f})")
    print(f" -> Day T Last 2 Bars vs Next Day Full Return:   r = {corr_full:.4f} (p-value: {p_corr_full:.4f})")
    
    # Test B: Directional Split Performance
    pos_ends = analysis_df[analysis_df['Predictor_Return'] > 0]
    neg_ends = analysis_df[analysis_df['Predictor_Return'] <= 0]
    
    print("\n[2] DIRECTIONAL SUB-GROUP ANALYSIS")
    print(f" Days ending Positive (N={len(pos_ends)}):")
    print(f"   * Avg Next-Day First 2 Bars Return: {pos_ends['Target_Early_Return'].mean()*100:.4f}%")
    print(f"   * Avg Next-Day Full Return:         {pos_ends['Target_Full_Return'].mean()*100:.4f}%")
    print(f" Days ending Negative (N={len(neg_ends)}):")
    print(f"   * Avg Next-Day First 2 Bars Return: {neg_ends['Target_Early_Return'].mean()*100:.4f}%")
    print(f"   * Avg Next-Day Full Return:         {neg_ends['Target_Full_Return'].mean()*100:.4f}%")
    
    t_stat_early, p_t_early = stats.ttest_ind(pos_ends['Target_Early_Return'], neg_ends['Target_Early_Return'], equal_var=False)
    t_stat_full, p_t_full = stats.ttest_ind(pos_ends['Target_Full_Return'], neg_ends['Target_Full_Return'], equal_var=False)
    
    print("\n[3] STATISTICAL SIGNIFICANCE TESTS (T-Test p-values)")
    print(f" -> Is the First 2 Bars difference significant? p-value = {p_t_early:.4f}")
    print(f" -> Is the Full Next Day difference significant?  p-value = {p_t_full:.4f}")
    
    return analysis_df

# Drop your file path right here:
# res = run_hypothesis_test('your_data_file.csv')

filtered_df = pd.DataFrame({'Currency': ['USDJPY', 'EURUSD', 'AUDUSD', 'US30.a']})

# Process loop
for currency in filtered_df['Currency']:
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    res = run_hypothesis_test(file_path_h4)

Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\USDJPY_h4.csv
Processing day-to-day transitions...

        HISTORICAL HYPOTHESIS TEST RESULTS        
Total trading day transitions analyzed: 1133

[1] LINEAR CORRELATION ANALYSIS (Pearson r)
 -> Day T Last 2 Bars vs Next Day First 2 Bars: r = -0.0031 (p-value: 0.9170)
 -> Day T Last 2 Bars vs Next Day Full Return:   r = 0.0241 (p-value: 0.4186)

[2] DIRECTIONAL SUB-GROUP ANALYSIS
 Days ending Positive (N=645):
   * Avg Next-Day First 2 Bars Return: 0.0315%
   * Avg Next-Day Full Return:         0.0690%
 Days ending Negative (N=488):
   * Avg Next-Day First 2 Bars Return: 0.0240%
   * Avg Next-Day Full Return:         0.0618%

[3] STATISTICAL SIGNIFICANCE TESTS (T-Test p-values)
 -> Is the First 2 Bars difference significant? p-value = 0.7146
 -> Is the Full Next Day difference significant?  p-value = 0.8514
Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\EURUSD_h4.csv
Processing day-to-day transitions...


In [6]:
# Tests for autocorrelation, directional trends, and relationships 
#     among the first 3 H4 bars of each trading day. Answer nothing strong
import pandas as pd
import numpy as np
import scipy.stats as stats

def run_intraday_structure_test(csv_path):
    """
    Tests for autocorrelation, directional trends, and relationships 
    among the first 3 H4 bars of each trading day.
    """
    print(f"Loading data from: {csv_path}")
    raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    
    # Structural positional mapping to avoid header errors
    df = pd.DataFrame()
    df['Date_Raw'] = raw_df.iloc[:, 0].astype(str)
    df['Time_Raw'] = raw_df.iloc[:, 1].astype(str)
    df['Open'] = raw_df.iloc[:, 2].astype(float)
    df['High'] = raw_df.iloc[:, 3].astype(float)
    df['Low'] = raw_df.iloc[:, 4].astype(float)
    df['Close'] = raw_df.iloc[:, 5].astype(float)
    
    # Handle volume if available in columns 6 or 7
    df['Volume'] = raw_df.iloc[:, 6].astype(float) if raw_df.shape[1] > 6 else 1.0

    # Unified Chronological Datetime Parsing
    df['Clean_Date'] = df['Date_Raw'].str.replace('.', '-', regex=False)
    df['Timestamp'] = pd.to_datetime(df['Clean_Date'] + ' ' + df['Time_Raw'])
    df = df.sort_values(by='Timestamp').reset_index(drop=True)
    
    # Group by calendar day
    df['Date_Group'] = df['Timestamp'].dt.date
    grouped = df.groupby('Date_Group')
    
    intraday_data = []
    
    print("Extracting morning session H4 bars...")
    for date, group in grouped:
        # We need at least the first 3 bars of the day present
        if len(group) >= 3:
            first_3 = group.head(3).reset_index(drop=True)
            
            # Calculate simple percentage returns for each individual bar
            r1 = (first_3['Close'].iloc[0] - first_3['Open'].iloc[0]) / first_3['Open'].iloc[0]
            r2 = (first_3['Close'].iloc[1] - first_3['Open'].iloc[1]) / first_3['Open'].iloc[1]
            r3 = (first_3['Close'].iloc[2] - first_3['Open'].iloc[2]) / first_3['Open'].iloc[2]
            
            intraday_data.append({
                'Date': date,
                'Bar1_Return': r1,
                'Bar2_Return': r2,
                'Bar3_Return': r3,
                'Bar1_Vol': first_3['Volume'].iloc[0],
                'Bar2_Vol': first_3['Volume'].iloc[1]
            })
            
    session_df = pd.DataFrame(intraday_data)
    
    if len(session_df) == 0:
        print("Execution halted: Insufficient data structure density.")
        return None
        
    print("\n" + "="*50)
    print("      INTRADAY H4 AUTOCORRELATION & TREND TEST      ")
    print("="*50)
    print(f"Total valid trading days analyzed: {len(session_df)}")
    
    # Test 1: Autocorrelation (Linear dependency)
    r1_r2_corr, p_12 = stats.pearsonr(session_df['Bar1_Return'], session_df['Bar2_Return'])
    r2_r3_corr, p_23 = stats.pearsonr(session_df['Bar2_Return'], session_df['Bar3_Return'])
    r1_r3_corr, p_13 = stats.pearsonr(session_df['Bar1_Return'], session_df['Bar3_Return'])
    
    print("\n[1] SEQUENTIAL AUTOCORRELATION (Pearson r)")
    print(f" -> Bar 1 vs Bar 2: r = {r1_r2_corr:.4f} (p-value: {p_12:.4f})")
    print(f" -> Bar 2 vs Bar 3: r = {r2_r3_corr:.4f} (p-value: {p_23:.4f})")
    print(f" -> Bar 1 vs Bar 3: r = {r1_r3_corr:.4f} (p-value: {p_13:.4f})")
    
    # Test 2: Directional Sequential Trends
    # Scenario A: What happens to Bar 3 if Bar 2 continued Bar 1's direction?
    matching_start = session_df[np.sign(session_df['Bar1_Return']) == np.sign(session_df['Bar2_Return'])]
    opposing_start = session_df[np.sign(session_df['Bar1_Return']) != np.sign(session_df['Bar2_Return'])]
    
    print("\n[2] DIRECTIONAL TREND ANALYSIS")
    print(f" Days where Bar 1 & Bar 2 moved in the SAME direction (N={len(matching_start)}):")
    print(f"   * Probability Bar 3 continues trend: {(matching_start['Bar3_Return'] * matching_start['Bar2_Return'] > 0).mean()*100:.2f}%")
    print(f"   * Avg Bar 3 Return:                  {matching_start['Bar3_Return'].mean()*100:.4f}%")
    
    print(f" Days where Bar 1 & Bar 2 moved in OPPOSITE directions (N={len(opposing_start)}):")
    print(f"   * Probability Bar 3 matches Bar 2:   {(opposing_start['Bar3_Return'] * opposing_start['Bar2_Return'] > 0).mean()*100:.2f}%")
    print(f"   * Avg Bar 3 Return:                  {opposing_start['Bar3_Return'].mean()*100:.4f}%")
    
    # Test 3: Sign/Directional Contingency (Chi-Square test on directions)
    b1_sign = np.sign(session_df['Bar1_Return']).replace(0, 1) # treat flat as positive for simplicity
    b2_sign = np.sign(session_df['Bar2_Return']).replace(0, 1)
    
    contingency_table = pd.crosstab(b1_sign, b2_sign)
    chi2, p_chi2, _, _ = stats.chi2_contingency(contingency_table)
    
    print("\n[3] CATEGORICAL DEPENDENCY TEST (Chi-Square)")
    print(f" -> Do Bar 1 and Bar 2 have a structural directional dependency? p-value = {p_chi2:.4f}")
    if p_chi2 < 0.05:
        print("    SUCCESS: Bar 2's direction is statistically dependent on Bar 1's direction.")
    else:
        print("    FAIL: Bar 1 and Bar 2's directions appear completely independent.")
        
    return session_df

# Execute across your loop using:

filtered_df = pd.DataFrame({'Currency': ['USDJPY', 'EURUSD', 'AUDUSD', 'US30.a']})

# Process loop
for currency in filtered_df['Currency']:
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    res = run_intraday_structure_test(file_path_h4)

Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\USDJPY_h4.csv
Extracting morning session H4 bars...

      INTRADAY H4 AUTOCORRELATION & TREND TEST      
Total valid trading days analyzed: 1137

[1] SEQUENTIAL AUTOCORRELATION (Pearson r)
 -> Bar 1 vs Bar 2: r = 0.0217 (p-value: 0.4645)
 -> Bar 2 vs Bar 3: r = 0.0647 (p-value: 0.0292)
 -> Bar 1 vs Bar 3: r = -0.0398 (p-value: 0.1804)

[2] DIRECTIONAL TREND ANALYSIS
 Days where Bar 1 & Bar 2 moved in the SAME direction (N=560):
   * Probability Bar 3 continues trend: 50.89%
   * Avg Bar 3 Return:                  0.0046%
 Days where Bar 1 & Bar 2 moved in OPPOSITE directions (N=577):
   * Probability Bar 3 matches Bar 2:   52.34%
   * Avg Bar 3 Return:                  0.0075%

[3] CATEGORICAL DEPENDENCY TEST (Chi-Square)
 -> Do Bar 1 and Bar 2 have a structural directional dependency? p-value = 0.3905
    FAIL: Bar 1 and Bar 2's directions appear completely independent.
Loading data from: C:\Users\nitis\Documents\Forex\D

In [10]:
import pandas as pd
import numpy as np
import scipy.stats as stats

def run_extreme_magnitude_test(csv_path, sigma_threshold=1.0):
    """
    Isolates days where Bar 2 experienced an extreme magnitude move 
    and translates the percentage sigma into an absolute asset price move ($ or pips).
    """
    print(f"Loading data from: {csv_path}")
    raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    
    # Structural positional mapping
    df = pd.DataFrame()
    df['Date_Raw'] = raw_df.iloc[:, 0].astype(str)
    df['Time_Raw'] = raw_df.iloc[:, 1].astype(str)
    df['Open'] = raw_df.iloc[:, 2].astype(float)
    df['Close'] = raw_df.iloc[:, 5].astype(float)

    df['Clean_Date'] = df['Date_Raw'].str.replace('.', '-', regex=False)
    df['Timestamp'] = pd.to_datetime(df['Clean_Date'] + ' ' + df['Time_Raw'])
    df = df.sort_values(by='Timestamp').reset_index(drop=True)
    
    df['Date_Group'] = df['Timestamp'].dt.date
    grouped = df.groupby('Date_Group')
    
    intraday_data = []
    for date, group in grouped:
        if len(group) >= 3:
            first_3 = group.head(3).reset_index(drop=True)
            r1 = (first_3['Close'].iloc[0] - first_3['Open'].iloc[0]) / first_3['Open'].iloc[0]
            r2 = (first_3['Close'].iloc[1] - first_3['Open'].iloc[1]) / first_3['Open'].iloc[1]
            r3 = (first_3['Close'].iloc[2] - first_3['Open'].iloc[2]) / first_3['Open'].iloc[2]
            
            # Absolute price movement of Bar 2 (Close minus Open)
            abs_move_2 = abs(first_3['Close'].iloc[1] - first_3['Open'].iloc[1])
            # Benchmark price level (using the Open of Bar 2)
            price_level_2 = first_3['Open'].iloc[1]
            
            intraday_data.append({
                'Date': date,
                'Bar1_Return': r1,
                'Bar2_Return': r2,
                'Bar3_Return': r3,
                'Bar2_Price_Level': price_level_2,
                'Bar2_Absolute_Move': abs_move_2
            })
            
    session_df = pd.DataFrame(intraday_data)
    
    if len(session_df) == 0:
        return None
        
    # Calculate the baseline volatility (standard deviation) of Bar 2 returns
    bar2_std = session_df['Bar2_Return'].std()
    
    # Define what constitutes an "extreme" percentage move
    threshold_pct = sigma_threshold * bar2_std
    
    # Translate percentage threshold into a representative price distance
    avg_price_level = session_df['Bar2_Price_Level'].iloc[-1]
    implied_absolute_move = avg_price_level * threshold_pct
    
    # Filter for extreme days
    extreme_days = session_df[session_df['Bar2_Return'].abs() > threshold_pct]
    normal_days = session_df[session_df['Bar2_Return'].abs() <= threshold_pct]
    
    print("\n" + "="*60)
    print(f"   MAGNITUDE EXHAUSTION TEST (Threshold: {sigma_threshold} Sigma)")
    print("="*60)
    print(f"Total baseline days: {len(session_df)}")
    print(f"Extreme Bar 2 days isolated: {len(extreme_days)} ({len(extreme_days)/len(session_df)*100:.1f}% of data)")
    
    # --- HERE IS THE TRANSLATION BREAKDOWN ---
    print(f"\n[VOLATILITY TRANSLATION]")
    print(f" -> Average Asset Price Level during Bar 2: {avg_price_level:.2f}")
    print(f" -> {sigma_threshold} Sigma Threshold (Percentage):    {threshold_pct*100:.3f}%")
    print(f" -> Implied 1-Sigma Move in Points/Dollars:   ±{implied_absolute_move:.4f}")
    print(f"    (i.e., Bar 2 must move more than {implied_absolute_move:.2f} points/units to trigger an event)")
    
    print(f"\n[1] ALL DAYS BACKGROUND")
    r_all, p_all = stats.pearsonr(session_df['Bar2_Return'], session_df['Bar3_Return'])
    print(f" -> Bar 2 vs Bar 3 Correlation: r = {r_all:.4f} (p-value: {p_all:.4f})")

    if len(extreme_days) > 5:
        print(f"\n[2] EXTREME DAYS ONLY (> {sigma_threshold} Std Dev)")
        r_ext, p_ext = stats.pearsonr(extreme_days['Bar2_Return'], extreme_days['Bar3_Return'])
        print(f" -> Bar 2 vs Bar 3 Correlation: r = {r_ext:.4f} (p-value: {p_ext:.4f})")
        
        reversion_count = ((extreme_days['Bar2_Return'] > 0) & (extreme_days['Bar3_Return'] < 0)) | \
                          ((extreme_days['Bar2_Return'] < 0) & (extreme_days['Bar3_Return'] > 0))
        reversion_rate = reversion_count.mean() * 100
        print(f" -> Absolute Reversion Win-Rate on Extremes: {reversion_rate:.2f}%")
    else:
        print("\n[2] EXTREME DAYS: Not enough data points matching this threshold.")
        
    if len(normal_days) > 5:
        print(f"\n[3] NORMAL/QUIET DAYS ONLY (<= {sigma_threshold} Std Dev)")
        r_norm, p_norm = stats.pearsonr(normal_days['Bar2_Return'], normal_days['Bar3_Return'])
        print(f" -> Bar 2 vs Bar 3 Correlation: r = {r_norm:.4f} (p-value: {p_norm:.4f})")
        
    return session_df

# Run it with 1.0 sigma first, then try tightening to 1.5 if you have enough data
# Process loop
for currency in filtered_df['Currency']:
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    run_extreme_magnitude_test(file_path_h4, sigma_threshold=1.0)

Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\USDJPY_h4.csv

   MAGNITUDE EXHAUSTION TEST (Threshold: 1.0 Sigma)
Total baseline days: 1137
Extreme Bar 2 days isolated: 205 (18.0% of data)

[VOLATILITY TRANSLATION]
 -> Average Asset Price Level during Bar 2: 160.35
 -> 1.0 Sigma Threshold (Percentage):    0.243%
 -> Implied 1-Sigma Move in Points/Dollars:   ±0.3903
    (i.e., Bar 2 must move more than 0.39 points/units to trigger an event)

[1] ALL DAYS BACKGROUND
 -> Bar 2 vs Bar 3 Correlation: r = 0.0647 (p-value: 0.0292)

[2] EXTREME DAYS ONLY (> 1.0 Std Dev)
 -> Bar 2 vs Bar 3 Correlation: r = 0.1133 (p-value: 0.1056)
 -> Absolute Reversion Win-Rate on Extremes: 46.34%

[3] NORMAL/QUIET DAYS ONLY (<= 1.0 Std Dev)
 -> Bar 2 vs Bar 3 Correlation: r = 0.0258 (p-value: 0.4322)
Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\EURUSD_h4.csv

   MAGNITUDE EXHAUSTION TEST (Threshold: 1.0 Sigma)
Total baseline days: 1137
Extreme Bar 2 days isolated: 276 (24.3

In [3]:
#If Bar 1 being BIG leads to mean reversion for H4
import pandas as pd
import numpy as np
import scipy.stats as stats

def run_buffered_sigma_test(csv_path, next_bar_buffer_sigma=0.25):
    print(f"Loading data from: {csv_path}")
    raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    
    # Structural positional mapping
    df = pd.DataFrame()
    df['Date_Raw'] = raw_df.iloc[:, 0].astype(str)
    df['Time_Raw'] = raw_df.iloc[:, 1].astype(str)
    df['Open'] = raw_df.iloc[:, 2].astype(float)
    df['Close'] = raw_df.iloc[:, 5].astype(float)

    # Clean date and create a continuous chronological datetime index
    df['Clean_Date'] = df['Date_Raw'].str.replace('.', '-', regex=False)
    df['Timestamp'] = pd.to_datetime(df['Clean_Date'] + ' ' + df['Time_Raw'])
    df = df.sort_values(by='Timestamp').reset_index(drop=True)
    
    # Calculate returns and map to the next bar
    df['Bar_Return'] = (df['Close'] - df['Open']) / df['Open']
    df['Next_Bar_Return'] = df['Bar_Return'].shift(-1)
    df = df.dropna().reset_index(drop=True)
    
    # Establish baseline volatility
    global_std = df['Bar_Return'].std()
    avg_price = df['Open'].mean()
    
    # Take absolute value of returns to measure pure magnitude in terms of Sigma
    df['Abs_Sigma'] = df['Bar_Return'].abs() / global_std
    
    # Define the minimum target return for the next bar to count as a "real" move
    buffer_pct_threshold = next_bar_buffer_sigma * global_std
    buffer_pts = buffer_pct_threshold * avg_price
    
    print("\n" + "="*60)
    print(f"   BUFFERED SIGMA TEST (3-BUCKET OUTCOMES WITH BAND RANGES)")
    print("="*60)
    print(f"Total bars analyzed: {len(df)}")
    print(f"Global 1-Sigma baseline (Percentage): {global_std*100:.3f}%")
    print(f"Average Asset Price Level: {avg_price:.2f}")
    print(f"Required Next-Bar Buffer ({next_bar_buffer_sigma} Sigma): ±{buffer_pts:.2f} points ({buffer_pct_threshold*100:.3f}%)")
    
    # Define our breakout bands
    bands = [
        (1.0, 2.0, "1.0x to 2.0x Sigma (Moderate Breakout)"),
        (2.0, 3.0, "2.0x to 3.0x Sigma (Strong Breakout)"),
        (3.0, float('inf'), "Over 3.0x Sigma (Extreme Exhaustion)")
    ]
    
    for lower, upper, label in bands:
        bucket_df = df[(df['Abs_Sigma'] >= lower) & (df['Abs_Sigma'] < upper)]
        
        # --- CALCULATE THE PHYSICAL POINT BOUNDARIES FOR THE BREAKOUT BARS ---
        lower_pts = lower * global_std * avg_price
        upper_pts = upper * global_std * avg_price
        pts_str = f"{lower_pts:.2f} to {upper_pts:.2f}" if upper != float('inf') else f"> {lower_pts:.2f}"
        
        print(f"\n" + "-"*60)
        print(f"BAND: {label}")
        print(f"-"*60)
        print(f" -> Breakout Range in Points:     {pts_str}")
        print(f" -> Occurrences found:            {len(bucket_df)} ({len(bucket_df)/len(df)*100:.2f}% of data)")
        
        if len(bucket_df) > 5:
            # 1. MEANINGFUL CONTINUATION: Next bar moves in same direction AND exceeds buffer
            continuation = ((bucket_df['Bar_Return'] > 0) & (bucket_df['Next_Bar_Return'] > buffer_pct_threshold)) | \
                           ((bucket_df['Bar_Return'] < 0) & (bucket_df['Next_Bar_Return'] < -buffer_pct_threshold))
            
            # 2. MEANINGFUL REVERSION: Next bar moves in opposite direction AND exceeds buffer
            reversion = ((bucket_df['Bar_Return'] > 0) & (bucket_df['Next_Bar_Return'] < -buffer_pct_threshold)) | \
                        ((bucket_df['Bar_Return'] < 0) & (bucket_df['Next_Bar_Return'] > buffer_pct_threshold))
            
            # 3. INDECISION / FLAT: Next bar fails to clear the buffer in either direction
            indecision = bucket_df['Next_Bar_Return'].abs() <= buffer_pct_threshold
            
            print(f" -> [Bucket 1] Meaningful Continuation: {continuation.mean() * 100:.2f}%")
            print(f" -> [Bucket 2] Meaningful Reversion:    {reversion.mean() * 100:.2f}%")
            print(f" -> [Bucket 3] Indecision / Flat Zone:  {indecision.mean() * 100:.2f}%")
        else:
            print(" -> Not enough data points to calculate buffered statistics.")
            
    return df
# Process loop
for currency in filtered_df['Currency']:
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    run_buffered_sigma_test(file_path_h4)

Loading data from: C:\Users\nitis\Documents\Forex\Data\New data\USDJPY_h4.csv

   BUFFERED SIGMA TEST (3-BUCKET OUTCOMES WITH BAND RANGES)
Total bars analyzed: 6824
Global 1-Sigma baseline (Percentage): 0.259%
Average Asset Price Level: 145.27
Required Next-Bar Buffer (0.25 Sigma): ±0.09 points (0.065%)

------------------------------------------------------------
BAND: 1.0x to 2.0x Sigma (Moderate Breakout)
------------------------------------------------------------
 -> Breakout Range in Points:     0.38 to 0.75
 -> Occurrences found:            1074 (15.74% of data)
 -> [Bucket 1] Meaningful Continuation: 38.64%
 -> [Bucket 2] Meaningful Reversion:    36.22%
 -> [Bucket 3] Indecision / Flat Zone:  25.14%

------------------------------------------------------------
BAND: 2.0x to 3.0x Sigma (Strong Breakout)
------------------------------------------------------------
 -> Breakout Range in Points:     0.75 to 1.13
 -> Occurrences found:            224 (3.28% of data)
 -> [Bucket 1] M

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats

def run_rolling_accumulation_test(csv_path, sigma_threshold=1.0):
    """
    Tests rolling 3-day accumulation regimes across overlapping periods.
    Evaluates mean reversion over the immediate subsequent 2-day and 5-day windows.
    """
    print(f"Loading daily data from: {csv_path}")
    raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    
    # 1. Structural Mapping & Chronological Sorting (Using correct 1 and 4 indices)
    df = pd.DataFrame()
    df['Date'] = pd.to_datetime(raw_df.iloc[:, 0].str.replace('.', '-', regex=False))
    df['Open'] = raw_df.iloc[:, 1].astype(float)
    df['Close'] = raw_df.iloc[:, 4].astype(float)
    df = df.sort_values(by='Date').reset_index(drop=True)
    df = df.drop_duplicates(subset=['Date']).reset_index(drop=True)
    
    # 2. Calculate Log Returns for mathematically accurate multi-day compounding
    df['Daily_Log_Return'] = np.log(df['Close'] / df['Open'])
    
    # 3. Rolling 3-Day Accumulation (The Independent Variable)
    # rolling(3).sum() adds up the log returns of the current day and the prior 2 days
    df['Accum_3D_Return'] = df['Daily_Log_Return'].rolling(window=3).sum()
    
    # 4. Map the Forward Target Windows (The Dependent Variables)
    # We use negative shifts to look forward into the future
    
    # Immediate next 2 days combined log return
    df['Next_2D_Return'] = df['Daily_Log_Return'].shift(-1) + df['Daily_Log_Return'].shift(-2)
    
    # Following week (next 5 days combined log return)
    df['Next_5D_Return'] = sum(df['Daily_Log_Return'].shift(-i) for i in range(1, 6))
    
    # Clean up edge rows that don't have enough history or future data
    df = df.dropna().reset_index(drop=True)
    
    # 5. Volatility Analysis on the 3-Day Cluster
    accum_std = df['Accum_3D_Return'].std()
    threshold_pct = sigma_threshold * accum_std
    
    # Isolate Regimes
    extreme_triggers = df[df['Accum_3D_Return'].abs() > threshold_pct]
    
    # Convert log returns back to standard percentages for the final output report
    std_accum_std = (np.exp(accum_std) - 1) * 100
    std_threshold_pct = (np.exp(threshold_pct) - 1) * 100
    
    # --- METRICS & OUTPUTS ---
    print("\n" + "="*75)
    print(f"   ROLLING OVERLAPPING ACCUMULATION TEST (3-Day Window vs Next 2D/5D)")
    print("="*75)
    print(f"Total trading days in dataset: {len(df)}")
    print(f"Extreme 3-Day clusters isolated: {len(extreme_triggers)} ({len(extreme_triggers)/len(df)*100:.1f}% of data)")
    
    print(f"\n[VOLATILITY STATS]")
    print(f" -> 3-Day Rolling Volatility (Std Dev): {std_accum_std:.3f}%")
    print(f" -> Isolation Threshold Target:          ±{std_threshold_pct:.3f}%")
    
    if len(extreme_triggers) > 5:
        # Reversion rate logic: did the forward window move opposite to the accumulation window?
        rev_2d_count = ((extreme_triggers['Accum_3D_Return'] > 0) & (extreme_triggers['Next_2D_Return'] < 0)) | \
                       ((extreme_triggers['Accum_3D_Return'] < 0) & (extreme_triggers['Next_2D_Return'] > 0))
        rev_rate_2d = rev_2d_count.mean() * 100
        
        rev_5d_count = ((extreme_triggers['Accum_3D_Return'] > 0) & (extreme_triggers['Next_5D_Return'] < 0)) | \
                       ((extreme_triggers['Accum_3D_Return'] < 0) & (extreme_triggers['Next_5D_Return'] > 0))
        rev_rate_5d = rev_5d_count.mean() * 100
        
        print(f"\n[1] IMMEDIATE REVERSION (Next 2 Trading Days)")
        r_2d, p_2d = stats.pearsonr(extreme_triggers['Accum_3D_Return'], extreme_triggers['Next_2D_Return'])
        print(f" -> Correlation: r = {r_2d:.4f} (p-value: {p_2d:.4f})")
        print(f" -> 2-Day Reversion Win-Rate: {rev_rate_2d:.2f}%")
        
        print(f"\n[2] FOLLOWING-WEEK REVERSION (Next 5 Trading Days)")
        r_5d, p_5d = stats.pearsonr(extreme_triggers['Accum_3D_Return'], extreme_triggers['Next_5D_Return'])
        print(f" -> Correlation: r = {r_5d:.4f} (p-value: {p_5d:.4f})")
        print(f" -> 5-Day Reversion Win-Rate: {rev_rate_5d:.2f}%")
    else:
        print("\n[!] INSIGNIFICANT SAMPLE: Try lowering the sigma threshold.")
    
    # Quick size diagnostic
    print(f"Avg move when Reversion Wins: {extreme_triggers[rev_2d_count]['Next_2D_Return'].abs().mean() * 100:.3f}%")
    print(f"Avg move when Reversion Fails: {extreme_triggers[~rev_2d_count]['Next_2D_Return'].abs().mean() * 100:.3f}%")
        
    return df

for currency in filtered_df['Currency']:
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}.csv"
    run_rolling_accumulation_test(file_path_h4, sigma_threshold=2.0)

Loading daily data from: C:\Users\nitis\Documents\Forex\Data\New data\USDJPY.csv

   ROLLING OVERLAPPING ACCUMULATION TEST (3-Day Window vs Next 2D/5D)
Total trading days in dataset: 1135
Extreme 3-Day clusters isolated: 59 (5.2% of data)

[VOLATILITY STATS]
 -> 3-Day Rolling Volatility (Std Dev): 1.098%
 -> Isolation Threshold Target:          ±2.208%

[1] IMMEDIATE REVERSION (Next 2 Trading Days)
 -> Correlation: r = -0.2083 (p-value: 0.1133)
 -> 2-Day Reversion Win-Rate: 64.41%

[2] FOLLOWING-WEEK REVERSION (Next 5 Trading Days)
 -> Correlation: r = -0.1995 (p-value: 0.1297)
 -> 5-Day Reversion Win-Rate: 61.02%
Avg move when Reversion Wins: 0.833%
Avg move when Reversion Fails: 0.907%
Loading daily data from: C:\Users\nitis\Documents\Forex\Data\New data\EURUSD.csv

   ROLLING OVERLAPPING ACCUMULATION TEST (3-Day Window vs Next 2D/5D)
Total trading days in dataset: 1135
Extreme 3-Day clusters isolated: 58 (5.1% of data)

[VOLATILITY STATS]
 -> 3-Day Rolling Volatility (Std Dev): 0.85